# Performance Replication: EKS primary → Minikube PR secondary

El primario usa AWS KMS auto-unseal. Al activar el secundario Shamir, Vault sustituye sus claves iniciales por la recovery key del primario.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find the persistent .env file")
load_dotenv(ENV_FILE, override=True)

required = (
    "VAULT_ADDR",
    "VAULT_TOKEN",
    "VAULT_CACERT",
    "VAULT_RECOVERY_KEY",
    "VAULT_PR_BOOTSTRAP_TOKEN",
    "VAULT_PR_TESTER_PASSWORD",
)
missing = [key for key in required if not os.getenv(key)]
if missing:
    raise RuntimeError(f"Missing required .env values: {', '.join(missing)}")

os.environ["ENV_FILE"] = str(ENV_FILE)
os.environ["WORKDIR"] = "/tmp/vault"
os.environ["VAULT_PR_WORKDIR"] = "/tmp/vault-pr"


## Configurar el primario

In [ ]:
%%bash
set -euo pipefail

vault status >/dev/null
if ! vault secrets list -format=json | jq -e 'has("secret-pr/")' >/dev/null; then
  vault secrets enable -path=secret-pr kv-v2
fi
vault kv put secret-pr/webapp/config foo=bar >/dev/null

vault policy write superuser - <<'EOF'
path "*" {
  capabilities = ["create", "read", "update", "delete", "list", "sudo"]
}
EOF

if ! vault auth list -format=json | jq -e 'has("userpass/")' >/dev/null; then
  vault auth enable userpass
fi

TESTER_PASSWORD="${VAULT_PR_TESTER_PASSWORD:-$(openssl rand -base64 24 | tr -d '\n')}"
vault write auth/userpass/users/tester \
  password="$TESTER_PASSWORD" \
  policies="superuser" >/dev/null

export TESTER_PASSWORD ENV_FILE
python3 - <<'PY'
import os
from pathlib import Path

path = Path(os.environ["ENV_FILE"])
key, value = "VAULT_PR_TESTER_PASSWORD", os.environ["TESTER_PASSWORD"]
lines = path.read_text().splitlines()
result, replaced = [], False
for line in lines:
    if line.split("=", 1)[0].strip() == key:
        result.append(f"{key}={value}")
        replaced = True
    else:
        result.append(line)
if not replaced:
    result.append(f"{key}={value}")
path.write_text("\n".join(result).rstrip() + "\n")
path.chmod(0o600)
PY
unset TESTER_PASSWORD


## Habilitar el primario y generar el token de activación

In [ ]:
%%bash
set -euo pipefail
mkdir -p "${WORKDIR}"
umask 077

MODE=$(vault read -format=json sys/replication/performance/status 2>/dev/null | jq -r '.data.mode // "disabled"')
if [[ "$MODE" == "disabled" ]]; then
  vault write -f sys/replication/performance/primary/enable \
    primary_cluster_addr=https://vault.jose-merchan.sbx.hashidemos.io:8201 >/dev/null
elif [[ "$MODE" != "primary" ]]; then
  echo "Unexpected primary replication mode: $MODE" >&2
  exit 1
fi

vault write -format=json \
  sys/replication/performance/primary/secondary-token \
  id="prp-$(date +%s)" | jq -r '.wrap_info.token' > "${WORKDIR}/pr_token.txt"
chmod 600 "${WORKDIR}/pr_token.txt"


## Activar y desellar el secundario local

In [ ]:
%%bash
set -euo pipefail

kubectl --context=PR get pod -n vaultpr vaultpr-0 >/dev/null
ACTIVATION_TOKEN=$(<"${WORKDIR}/pr_token.txt")

kubectl --context=PR exec -n vaultpr vaultpr-0 -- \
  env VAULT_SKIP_VERIFY=true VAULT_TOKEN="$VAULT_PR_BOOTSTRAP_TOKEN" vault write \
    sys/replication/performance/secondary/enable \
    primary_api_addr="$VAULT_ADDR" \
    token="$ACTIVATION_TOKEN"
unset ACTIVATION_TOKEN

# Replication changes the secondary seal. Restart every pod and unseal it with
# the primary recovery key, which becomes the secondary Shamir unseal key.
kubectl --context=PR delete pods -n vaultpr vaultpr-0 vaultpr-1 vaultpr-2 --wait=false
for pod in vaultpr-0 vaultpr-1 vaultpr-2; do
  for attempt in $(seq 1 120); do
    vault_status=$(kubectl --context=PR exec -n vaultpr "$pod" -- \
      env VAULT_SKIP_VERIFY=true vault status -format=json 2>/dev/null || true)
    initialized=$(jq -r '.initialized // false' <<<"$vault_status" 2>/dev/null || echo false)
    sealed=$(jq -r '.sealed // true' <<<"$vault_status" 2>/dev/null || echo true)
    [[ "$initialized" == "true" ]] && break
    [[ "$attempt" == "120" ]] && { echo "$pod did not initialize" >&2; exit 1; }
    sleep 3
  done
  if [[ "$sealed" == "true" ]]; then
    kubectl --context=PR exec -n vaultpr "$pod" -- \
      env VAULT_SKIP_VERIFY=true vault operator unseal "$VAULT_RECOVERY_KEY" >/dev/null
  fi
done

for attempt in $(seq 1 120); do
  ready=$(kubectl --context=PR get statefulset vaultpr -n vaultpr -o jsonpath='{.status.readyReplicas}')
  [[ "${ready:-0}" == "3" ]] && break
  [[ "$attempt" == "120" ]] && exit 1
  sleep 5
done


## Validar la replicación y el acceso

In [ ]:
%%bash
set -euo pipefail

TESTER_LOGIN=$(kubectl --context=PR exec -n vaultpr vaultpr-0 -- \
  env VAULT_SKIP_VERIFY=true vault login -format=json \
    -method=userpass username=tester password="$VAULT_PR_TESTER_PASSWORD")
SECONDARY_TOKEN=$(jq -r '.auth.client_token' <<<"$TESTER_LOGIN")
unset TESTER_LOGIN

if ! kubectl --context=PR exec -n vaultpr vaultpr-0 -- \
  env VAULT_SKIP_VERIFY=true VAULT_TOKEN="$SECONDARY_TOKEN" \
  vault audit list -format=json | jq -e 'has("file/")' >/dev/null; then
  kubectl --context=PR exec -n vaultpr vaultpr-0 -- \
    env VAULT_SKIP_VERIFY=true VAULT_TOKEN="$SECONDARY_TOKEN" \
    vault audit enable -local -path=file file file_path=/vault/audit/vault.log
fi

kubectl --context=PR exec -n vaultpr vaultpr-0 -- \
  env VAULT_SKIP_VERIFY=true VAULT_TOKEN="$SECONDARY_TOKEN" \
  vault read -format=json sys/replication/performance/status | \
  jq -e '.data.mode == "secondary" and .data.connection_state == "ready"' >/dev/null

kubectl --context=PR exec -n vaultpr vaultpr-0 -- \
  env VAULT_SKIP_VERIFY=true VAULT_TOKEN="$SECONDARY_TOKEN" vault kv get \
    -field=foo secret-pr/webapp/config | grep -qx bar
unset SECONDARY_TOKEN

echo "Performance secondary is operational and replicated data is readable."
